dataset :- https://www.kaggle.com/datasets/ameythakur20/asl-alphabet-train

In [ ]:
!pip install ultralytics kagglehub scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 63.6 MB/s eta 0:00:00


In [ ]:
import kagglehub
from pathlib import Path

path = kagglehub.dataset_download("ameythakur20/asl-alphabet-train")
dataset_path = Path(path)

print(dataset_path)

100%|██████████| 12.2M/12.2M [00:02<00:00, 4.70MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/ameythakur20/asl-alphabet-train/versions/1


In [ ]:
import os
import random
import shutil
from sklearn.model_selection import train_test_split
from pathlib import Path # Explicitly import Path here for clarity and robustness

# Fix 1: Adjust dataset_path to point to the actual folder containing class directories (A, B, C, ...)
# The 'path' variable from kagglehub.dataset_download points to '...versions/1',
# and the actual dataset's class folders are within a subfolder named 'asl-alphabet-train'.
dataset_path = Path(path) / "asl-alphabet-train"

output_path = Path("/content/asl_yolo")

images_train = output_path / "train/images"
labels_train = output_path / "train/labels"
images_val = output_path / "valid/images"
labels_val = output_path / "valid/labels"

for p in [images_train, labels_train, images_val, labels_val]:
    p.mkdir(parents=True, exist_ok=True)

# ❌ remove bad classes
REMOVE_CLASSES = ["nothing", "space", "del"]

# Now, os.listdir(dataset_path) will correctly list 'A', 'B', 'C', 'del', 'nothing', 'space', etc.
classes = sorted([
    cls_name for cls_name in os.listdir(dataset_path) # Use cls_name for clarity
    if cls_name.lower() not in REMOVE_CLASSES
])

class_map = {cls_name: i for i, cls_name in enumerate(classes)}

all_images = []

# ✅ collect images safely
for cls_name in classes: # cls_name will be 'A', 'B', 'C', etc.
    cls_folder = dataset_path / cls_name # e.g., Path('/root/.cache/kagglehub/datasets/ameythakur20/asl-alphabet-train/versions/1/asl-alphabet-train/A')
    for img_file_name in os.listdir(cls_folder): # img_file_name will be actual image filenames like 'A1.jpg'
        img_path_full = cls_folder / img_file_name # img_path_full will be Path('/root/.../A/A1.jpg')

        if img_path_full.is_file() and img_file_name.lower().endswith(('.jpg', '.png', '.jpeg')):
            all_images.append((cls_name, img_path_full))

print("Total images before balance:", len(all_images))

# 🔥 BALANCING (VERY IMPORTANT)
MAX_IMAGES_PER_CLASS = 1200

balanced_data = []
class_count = {}

for cls, img_path_obj in all_images: # Iterate over (class_name, Path_to_image_file) tuples
    if class_count.get(cls, 0) < MAX_IMAGES_PER_CLASS:
        balanced_data.append((cls, img_path_obj))
        class_count[cls] = class_count.get(cls, 0) + 1

all_images = balanced_data

print("Total images after balance:", len(all_images))

# split
train_data, val_data = train_test_split(all_images, test_size=0.2, random_state=42)

def process_data(data, img_folder, label_folder):
    for cls_name, img_path_obj in data: # cls_name is 'A', 'B', etc.; img_path_obj is the Path object for the image file
        img_name = img_path_obj.name

        shutil.copy(img_path_obj, img_folder / img_name)

        class_id = class_map[cls_name]
        label_path = label_folder / (img_name.rsplit('.', 1)[0] + ".txt")

        # full image bbox
        with open(label_path, "w") as f:
            f.write(f"{class_id} 0.5 0.5 1.0 1.0\n")

process_data(train_data, images_train, labels_train)
process_data(val_data, images_val, labels_val)

print("Dataset ready ✅")

Total images before balance: 780
Total images after balance: 780
Dataset ready ✅


In [ ]:
yaml_path = "/content/asl_yolo/data.yaml"

with open(yaml_path, "w") as f:
    f.write("path: /content/asl_yolo\n")
    f.write("train: train/images\n")
    f.write("val: valid/images\n")
    f.write("names:\n")

    for cls, idx in class_map.items():
        f.write(f"  {idx}: {cls}\n")

print("YAML created ✅")

YAML created ✅


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8m.pt")  # 🔥 better than nano

model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=32,
    patience=10,

    # 🔥 augmentations
    degrees=15,
    translate=0.1,
    scale=0.5,
    shear=5,
    mosaic=1.0,
    mixup=0.2,

    # ⚠️ IMPORTANT
    fliplr=0.0,
    flipud=0.0
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.30 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/asl_yolo/data.yaml, degrees=15, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e3c70475c40>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.04104

In [ ]:
from google.colab import files

files.download("/content/runs/detect/train/weights/best.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>